# 🫁 ViT Chest X-Ray TB Detection — Attention Heatmap Pipeline

This notebook loads a fine-tuned Vision Transformer (ViT) model to classify Chest X-Rays as **TB** or **Healthy** and generates interpretable **attention-based heatmaps** showing which pulmonary regions drove the prediction.

## Structure
| Cell | Description |
|------|-------------|
| 1 | Imports & device setup |
| 2 | `load_class_mapping()` |
| 3 | `load_model()` |
| 4 | `preprocess_image()` |
| 5 | `predict()` |
| 6 | `extract_attention()` |
| 7 | `generate_heatmap()` |
| 8 | `get_region()` |
| **9** | **⚡ TEST CELL — 5 healthy + 5 TB images** |

In [ ]:
%pip install torch torchvision transformers opencv-python pillow kagglehub matplotlib

In [2]:
# ============================================================
# Cell 1 — Imports & Device Setup
# ============================================================
import os
import json
import random
import glob

import numpy as np
import cv2
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from transformers import ViTForImageClassification

# ----------------------------------------------------------
# Use GPU when available; otherwise fall back to CPU
# ----------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Running on device: {device}')

✅ Running on device: cpu


In [3]:
# ============================================================
# Cell 2 — load_class_mapping()
# ============================================================

def load_class_mapping(mapping_path: str) -> dict:
    """
    Load class_to_idx JSON and invert it to idx → class_name.

    Supports two formats:
      - {"health": 0, "tb": 1}   ← class → index  (typical torchvision output)
      - {"0": "health", "1": "tb"} ← index → class  (already inverted)

    Returns
    -------
    dict : {int index: str label}
    """
    with open(mapping_path, 'r') as f:
        raw = json.load(f)

    # Detect format: if all values are ints → class_to_idx format
    if all(isinstance(v, int) for v in raw.values()):
        idx_to_class = {v: k for k, v in raw.items()}
    else:
        # Already idx_to_class; just coerce keys to int
        idx_to_class = {int(k): v for k, v in raw.items()}

    print(f'📋 Class mapping loaded: {idx_to_class}')
    return idx_to_class

In [4]:
# ============================================================
# Cell 3 — load_model()
# ============================================================

def load_model(weights_path: str, num_labels: int = 2) -> ViTForImageClassification:
    """
    Build a ViTForImageClassification model backed by
    'google/vit-base-patch16-224', replace its head with
    a linear layer for `num_labels` classes, and load the
    fine-tuned weights from `weights_path`.

    Parameters
    ----------
    weights_path : str
        Path to the .pth file saved during training.
    num_labels : int
        Number of output classes (default 2: TB / Healthy).

    Returns
    -------
    ViTForImageClassification in eval mode, on `device`.
    """
    # ----------------------------------------------------------
    # 1. Init the HuggingFace ViT and override the classifier
    #    `ignore_mismatched_sizes=True` prevents an error when
    #     the pretrained head (1000-class ImageNet) differs from
    #     our fine-tuned head (2-class TB detector).
    # ----------------------------------------------------------
    model = ViTForImageClassification.from_pretrained(
        'google/vit-base-patch16-224',
        num_labels=num_labels,
        ignore_mismatched_sizes=True,
        output_attentions=True   # <-- essential for heatmap extraction
    )

    # ----------------------------------------------------------
    # 2. Load fine-tuned weights
    # ----------------------------------------------------------
    checkpoint = torch.load(weights_path, map_location=device)

    # Handle checkpoints that wrap state_dict in an outer dict
    state_dict = checkpoint.get('state_dict', checkpoint)

    # `strict=False` tolerates minor key mismatches (e.g. extra
    # buffers saved during training that are not needed at inference)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f'⚠️  Missing keys (not loaded): {missing[:5]} ...')
    if unexpected:
        print(f'⚠️  Unexpected keys (ignored): {unexpected[:5]} ...')

    model.to(device)
    model.eval()   # Disable dropout / BatchNorm training behaviour
    print(f'✅ Model loaded from "{weights_path}" and set to eval mode.')
    return model

In [5]:
# ============================================================
# Cell 4 — preprocess_image()
# ============================================================

# ImageNet normalisation constants used during ViT pre-training
_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD  = [0.229, 0.224, 0.225]

_transform = transforms.Compose([
    transforms.Resize((224, 224)),   # ViT base expects 224x224 input
    transforms.ToTensor(),           # HWC uint8 → CHW float32 in [0,1]
    transforms.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD)
])


def preprocess_image(image_path: str):
    """
    Load a .jpg or .png chest X-ray image from disk,
    apply the standard ViT preprocessing pipeline, and
    return both the model-ready tensor and the raw NumPy
    array (needed for heatmap overlay).

    Returns
    -------
    input_tensor : torch.Tensor  shape (1, 3, 224, 224), on `device`
    image_np     : np.ndarray    shape (H, W, 3), uint8 RGB
    """
    if not os.path.isfile(image_path):
        raise FileNotFoundError(f'Image not found: {image_path}')

    pil_image = Image.open(image_path).convert('RGB')
    image_np  = np.array(pil_image)               # Keep original for overlay
    input_tensor = _transform(pil_image).unsqueeze(0).to(device)

    return input_tensor, image_np

In [6]:
# ============================================================
# Cell 5 — predict()
# ============================================================

def predict(model: ViTForImageClassification, input_tensor: torch.Tensor):
    """
    Run a single forward pass through the ViT model.

    Returns
    -------
    pred_idx : int    — predicted class index
    conf     : float  — confidence probability in [0, 1]
    outputs  : ViTImageClassifierOutput — contains logits & attentions
    """
    with torch.no_grad():   # Inference only; no gradients needed
        outputs = model(pixel_values=input_tensor)

    # Convert raw logits → class probabilities via Softmax
    probs = F.softmax(outputs.logits, dim=1)   # shape: (1, num_labels)

    # Extract the winning class and its probability
    conf, pred_idx = torch.max(probs, dim=1)

    return pred_idx.item(), conf.item(), outputs

In [7]:
# ============================================================
# Cell 6 — extract_attention()
# ============================================================

def extract_attention(outputs) -> np.ndarray:
    """
    Derive a 2-D spatial attention map from the model's last
    transformer layer.

    Steps
    -----
    1. Grab attention tensors from the LAST encoder layer.
       Shape: (batch=1, heads=12, seq_len=197, seq_len=197)
       (seq_len = 1 CLS token + 14×14 = 196 image patches)

    2. Average across all 12 attention heads.
       Shape: (1, 197, 197)

    3. Extract only the row corresponding to the CLS token
       (index 0) attending TO patch tokens (indices 1→196).
       Shape: (196,)

    4. Reshape to the 14×14 patch grid.

    5. Min-max normalise to [0, 1].

    Returns
    -------
    np.ndarray of shape (14, 14), float32, in [0, 1].
    """
    # Step 1: last layer, shape (1, 12, 197, 197)
    last_attn = outputs.attentions[-1]

    # Step 2: mean over heads → (1, 197, 197)
    avg_attn = last_attn.mean(dim=1)

    # Step 3: CLS row, patches only → (196,)
    cls_attn = avg_attn[0, 0, 1:]   # skip CLS-to-CLS self-attention

    # Step 4: 196 → 14×14
    grid = int(cls_attn.numel() ** 0.5)   # 14 for a 224/16 model
    attn_map = cls_attn.reshape(grid, grid).cpu().numpy()

    # Step 5: Normalise
    lo, hi = attn_map.min(), attn_map.max()
    if hi - lo > 1e-8:            # Guard against divide-by-zero
        attn_map = (attn_map - lo) / (hi - lo)
    else:
        attn_map = np.zeros_like(attn_map)

    return attn_map.astype(np.float32)

In [8]:
# ============================================================
# Cell 7 — generate_heatmap()
# ============================================================

def generate_heatmap(
    image_np: np.ndarray,
    attention_map: np.ndarray,
    save_path: str = None,
    alpha: float = 0.45
) -> np.ndarray:
    """
    Overlay a coloured attention heatmap onto the original X-ray.

    Parameters
    ----------
    image_np     : RGB uint8 array (any H×W)
    attention_map: float32 array in [0, 1], typically 14×14
    save_path    : if given, the overlay is saved to this path
    alpha        : heatmap opacity (0 = invisible, 1 = fully opaque)

    Returns
    -------
    overlay_rgb : np.ndarray — blended image in RGB uint8
    """
    H, W = image_np.shape[:2]

    # ----------------------------------------------------------
    # Upsample: 14×14 → original image resolution
    # INTER_CUBIC gives smoother gradients than nearest-neighbour
    # ----------------------------------------------------------
    attn_resized = cv2.resize(attention_map, (W, H),
                              interpolation=cv2.INTER_CUBIC)

    # Scale to [0, 255] and apply the JET colormap (blue→red spectrum)
    heatmap_u8 = np.uint8(255 * attn_resized)
    heatmap_bgr = cv2.applyColorMap(heatmap_u8, cv2.COLORMAP_JET)

    # OpenCV works in BGR; convert original image accordingly
    image_bgr = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)

    # Weighted blend: (1-alpha)*original + alpha*heatmap
    overlay_bgr = cv2.addWeighted(image_bgr, 1 - alpha,
                                  heatmap_bgr, alpha, 0)

    # Convert back to RGB for matplotlib display
    overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)

    if save_path:
        # plt.imsave expects RGB, which is what we already have
        plt.imsave(save_path, overlay_rgb)
        print(f'   💾 Heatmap saved → {save_path}')

    return overlay_rgb

In [9]:
# ============================================================
# Cell 8 — get_region()
# ============================================================

def get_region(attention_map: np.ndarray) -> str:
    """
    Divide the 14×14 attention grid into 6 anatomical regions
    and return the name of the most attended region.

    Layout
    ------
      columns split at col_mid  → left / right
      rows    split at 1/3, 2/3 → upper / middle / lower

    Returns
    -------
    str : e.g. 'upper left lung'
    """
    gh, gw = attention_map.shape     # typically (14, 14)

    c = gw // 2          # left | right boundary
    r1 = gh // 3         # upper | middle boundary
    r2 = 2 * (gh // 3)  # middle | lower boundary

    regions = {
        'upper left' : attention_map[:r1,  :c ].mean(),
        'upper right': attention_map[:r1,   c:].mean(),
        'middle left': attention_map[r1:r2, :c].mean(),
        'middle right':attention_map[r1:r2,  c:].mean(),
        'lower left' : attention_map[r2:,   :c].mean(),
        'lower right': attention_map[r2:,    c:].mean(),
    }

    best = max(regions, key=regions.get)
    return f'{best} lung'

---
## ⚡ Cell 9 — Testing Cell
> **Run ALL cells above first.** This standalone cell downloads the Kaggle dataset,
> picks 5 healthy + 5 TB images at random, and runs the complete pipeline on each.

In [14]:
# ============================================================
# Cell 9 — TESTING (do not mix with pipeline functions above)
# ============================================================
#D:\SEM8\AIH\Mini-Project\vit_tb_model.pth
import kagglehub   # pip install kagglehub

# ------------------------------------------------------------------
# 9.1  Paths to model artefacts
#      Adjust BASE_FOLDER if your weights live elsewhere.
# ------------------------------------------------------------------

BASE_FOLDER = 'D:/SEM8/AIH/Mini-Project'
MODEL_PATH   = os.path.join(BASE_FOLDER, 'vit_tb_model.pth')
MAPPING_PATH = os.path.join(BASE_FOLDER, 'class_mapping.json')

for p in (MODEL_PATH, MAPPING_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(
            f'Required file not found: {p}\n'
            f'Make sure "{BASE_FOLDER}/" contains '
            f'vit_tb_model.pth and class_mapping.json'
        )

# ------------------------------------------------------------------
# 9.2  Download / locate cached Kaggle dataset
# ------------------------------------------------------------------
print('📥 Locating Kaggle dataset (downloads once, then cached) ...')
dataset_root = kagglehub.dataset_download('tirthachhetri/tuberclosis-dataset')
print(f'📂 Dataset root: {dataset_root}')

HEALTH_DIR = os.path.join(dataset_root, 'TB_Split', 'valid', 'health')
TB_DIR     = os.path.join(dataset_root, 'TB_Split', 'valid', 'tb')

for d in (HEALTH_DIR, TB_DIR):
    if not os.path.isdir(d):
        raise FileNotFoundError(
            f'Expected directory not found: {d}\n'
            f'Verify that the Kaggle dataset contains TB_Split/valid/health and tb/'
        )

# ------------------------------------------------------------------
# 9.3  Sample images
# ------------------------------------------------------------------
def _collect_images(folder: str) -> list:
    """Return all .jpg/.png paths inside `folder`."""
    return (
        glob.glob(os.path.join(folder, '*.jpg')) +
        glob.glob(os.path.join(folder, '*.JPG')) +
        glob.glob(os.path.join(folder, '*.png')) +
        glob.glob(os.path.join(folder, '*.PNG'))
    )

health_pool = _collect_images(HEALTH_DIR)
tb_pool     = _collect_images(TB_DIR)

print(f'🔎 Found {len(health_pool)} healthy and {len(tb_pool)} TB images in validation set.')

N = 5   # images per class
random.seed(42)   # reproducible sampling
test_images = random.sample(health_pool, min(N, len(health_pool))) + \
              random.sample(tb_pool,     min(N, len(tb_pool)))
random.shuffle(test_images)

# ------------------------------------------------------------------
# 9.4  Initialise model (once — reused for all 10 images)
# ------------------------------------------------------------------
print('\n🧠 Loading model ...')
idx_to_class = load_class_mapping(MAPPING_PATH)
model        = load_model(MODEL_PATH, num_labels=len(idx_to_class))

# Output directory for saved heatmaps
os.makedirs('heatmap_outputs', exist_ok=True)

# ------------------------------------------------------------------
# 9.5  Run pipeline on each image and display results
# ------------------------------------------------------------------
print('\n' + '='*60)
print('   INFERENCE RESULTS')
print('='*60)

for i, img_path in enumerate(test_images, start=1):
    img_name = os.path.basename(img_path)

    # ── Preprocessing ──────────────────────────────────────────
    input_tensor, original_np = preprocess_image(img_path)

    # ── Inference ──────────────────────────────────────────────
    pred_idx, conf, outputs = predict(model, input_tensor)
    label = idx_to_class.get(pred_idx, f'Class_{pred_idx}')

    # ── Attention map ──────────────────────────────────────────
    attn_map   = extract_attention(outputs)
    top_region = get_region(attn_map)

    # ── Heatmap overlay ────────────────────────────────────────
    save_path  = os.path.join('heatmap_outputs', f'heatmap_{i:02d}_{img_name}')
    overlay    = generate_heatmap(original_np, attn_map, save_path=save_path)

    # ── Console output ─────────────────────────────────────────
    print(f'\n[{i:02d}/10] {img_name}')
    print(f'   Prediction : {label.upper()}')
    print(f'   Confidence : {conf * 100:.2f}%')
    print(f'   Region     : {top_region}')

    # ── Side-by-side figure ────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    fig.suptitle(
        f'[{i:02d}/10]  {img_name}\n'
        f'Prediction: {label.upper()}  |  '
        f'Confidence: {conf * 100:.1f}%  |  '
        f'Region: {top_region}',
        fontsize=11, fontweight='bold'
    )

    axes[0].imshow(original_np)
    axes[0].set_title('Original Chest X-Ray')
    axes[0].axis('off')

    axes[1].imshow(overlay)
    axes[1].set_title('Attention Heatmap Overlay')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

print('\n' + '='*60)
print('✅  All 10 images processed.  Heatmaps saved in heatmap_outputs/')
print('='*60)

📥 Locating Kaggle dataset (downloads once, then cached) ...


  4%|▍         | 277M/6.68G [01:28<34:53, 3.29MB/s]   


KeyboardInterrupt: 